# Training DL-AMC (1D ResNet) Directly on Raw IQ Signals

This notebook provides an accelerated **Deep Learning Automatic Modulation Classification (DL-AMC)** training pipeline using a **1D ResNet** operating directly on complex baseband IQ data `[Batch, 2 (I, Q), SequenceLength]`.

## Pipeline Architecture
1. **Configurable Dataset Selection**: Choose between **RadioML 2018.01A** (real RF impairments across 24 classes) or **High-Throughput Synthetic IQ** (30 classes).
2. **Out-of-Core Streaming DataLoader**: Eliminates RAM bloat and Out-Of-Memory (OOM) crashes by reading slices from HDF5 on-the-fly or generating stochastic IQ batches dynamically.
3. **Unified Dataset Length Control**: Single parameter controlling sample allocation across all modulation classes and SNR levels.
4. **1D ResNet Backbone**: 4 residual stages, batch normalization, global average pooling, and dynamic batch/sequence-length support.
5. **GPU Acceleration & Automatic Mixed Precision (AMP)**: FP16 tensor core acceleration with AdamW and Cosine Annealing.
6. **Production Artifact Export**: Portable ONNX runtime export (`amc_resnet1d.onnx`) with automatic checkpoint cleanup.


## 1. Tunable Configuration & Dataset Selection

Configure the dataset source, dataset length (number of samples per class), and training hyperparameters here.


In [ ]:
# ==============================================================================
# CONFIGURATION BLOCK
# ==============================================================================
import os
import sys

# ------------------------------------------------------------------------------
# 1. Dataset Selection
# ------------------------------------------------------------------------------
# Choose between:
#   - "RADIOML"   : Uses RadioML 2018.01A HDF5 dataset (24 classes, 26 SNR steps)
#   - "GENERATED" : Uses high-throughput on-the-fly synthetic IQ generator (30 classes)
DATASET_CHOICE = "RADIOML"

# ------------------------------------------------------------------------------
# 2. Dataset Length / Sample Limit
# ------------------------------------------------------------------------------
# Controls dataset size across both "RADIOML" and "GENERATED" pipelines:
# - For "RADIOML": Specifies frames per modulation class across all 26 SNRs.
#     The frames are divided evenly across SNR levels (-20 dB to +30 dB in 2 dB steps).
#     Examples:
#       * SAMPLES_PER_CLASS = 2600  -> 100 frames/SNR level (62,400 total frames)
#       * SAMPLES_PER_CLASS = 5200  -> 200 frames/SNR level (124,800 total frames)
#       * SAMPLES_PER_CLASS = None  -> Full dataset: all 4,096 frames/SNR (2,555,904 frames)
# - For "GENERATED": Specifies number of synthetic frames synthesized per class.
#     Examples:
#       * SAMPLES_PER_CLASS = 1000  -> 30 classes * 1000 = 30,000 total frames
#       * SAMPLES_PER_CLASS = 2000  -> 30 classes * 2000 = 60,000 total frames
SAMPLES_PER_CLASS = 2600  # Set to None for full RadioML dataset, or an integer for faster training

# ------------------------------------------------------------------------------
# 3. Model & Training Hyperparameters
# ------------------------------------------------------------------------------
FRAME_LEN = 1024          # Sequence length of raw IQ samples per frame
BATCH_SIZE = 128          # Mini-batch size for DataLoader
EPOCHS = 35               # Total training epochs
LEARNING_RATE = 2e-3      # Initial learning rate for AdamW optimizer
WEIGHT_DECAY = 1e-4       # Weight decay (L2 regularization)
NUM_WORKERS = 2           # Number of parallel worker threads for streaming DataLoader
VAL_SPLIT = 0.20          # Validation split ratio (stratified across classes)
RANDOM_SEED = 42          # Reproducibility seed

# ------------------------------------------------------------------------------
# 4. Dataset Availability Check & Validation
# ------------------------------------------------------------------------------
def locate_radioml_dataset():
    """Search for RadioML 2018.01A HDF5 file in Kaggle and local directories."""
    candidate_paths = [
        "/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018/data/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018/data/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018.01a/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018.01a/GOLD_XYZ_OSC.0001_1024.hdf5",
        "GOLD_XYZ_OSHF.hdf5",
        "GOLD_XYZ_OSC.0001_1024.hdf5",
        "../data/GOLD_XYZ_OSHF.hdf5",
        "data/GOLD_XYZ_OSHF.hdf5",
        "/mnt/wksp/kaggle_5dag/experiments/gr-playground/data/GOLD_XYZ_OSHF.hdf5",
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    return None

DATASET_CHOICE_UPPER = DATASET_CHOICE.strip().upper()
if DATASET_CHOICE_UPPER == "RADIOML":
    dataset_path = locate_radioml_dataset()
    if dataset_path is None:
        raise FileNotFoundError(
            "\n" + "=" * 80 + "\n"
            "ERROR: DATASET_CHOICE is set to 'RADIOML', but RadioML 2018 HDF5 file "
            "(GOLD_XYZ_OSHF.hdf5 or GOLD_XYZ_OSC.0001_1024.hdf5) was not found!\n\n"
            "Checked locations:\n"
            "  - /kaggle/input/datasets/pinxau1000/radioml2018/...\n"
            "  - /kaggle/input/radioml2018/...\n"
            "  - /kaggle/input/radioml2018.01a/...\n"
            "  - Current working directory and ./data/\n\n"
            "Action Required:\n"
            "  1. Add dataset 'pinxau1000/radioml2018' to your Kaggle Notebook input, OR\n"
            "  2. Set DATASET_CHOICE = 'GENERATED' in the configuration block to train on synthetic IQ signals.\n"
            + "=" * 80
        )
    print(f"[CONFIG] Dataset mode: RADIOML 2018.01A (Path: {dataset_path})")
elif DATASET_CHOICE_UPPER == "GENERATED":
    dataset_path = None
    print("[CONFIG] Dataset mode: GENERATED (High-throughput synthetic IQ streaming)")
else:
    raise ValueError(f"Invalid DATASET_CHOICE: '{DATASET_CHOICE}'. Allowed options: 'RADIOML', 'GENERATED'.")

print(f"[CONFIG] SAMPLES_PER_CLASS : {SAMPLES_PER_CLASS if SAMPLES_PER_CLASS is not None else 'FULL DATASET (All 4,096 per SNR)'}")
print(f"[CONFIG] Batch Size        : {BATCH_SIZE}")
print(f"[CONFIG] Training Epochs   : {EPOCHS}")
print(f"[CONFIG] Learning Rate     : {LEARNING_RATE}")
print(f"[CONFIG] Frame Length      : {FRAME_LEN} samples")


In [ ]:
# 2. Environment Setup & Dependency Verification
import subprocess

print("Python version:", sys.version)

# Matplotlib cache setup for headless / sandbox runs
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib_cache")
os.makedirs("/tmp/matplotlib_cache", exist_ok=True)

# Required packages
required_pkgs = ["onnx", "onnxruntime", "seaborn", "tqdm", "scikit-learn", "pandas", "scipy"]
if DATASET_CHOICE_UPPER == "RADIOML":
    required_pkgs.append("h5py")

for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
        except Exception as e:
            print(f"Warning: automatic pip install failed ({e}). Proceeding if package is optional.")

import numpy as np
if DATASET_CHOICE_UPPER == "RADIOML":
    import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm.auto import tqdm

# Set plotting style
try:
    import seaborn as sns
    sns_available = True
    plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
except ImportError:
    sns_available = False

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Set random seed
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Select compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if device.type == "cuda":
    print(f"GPU Accelerator: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
else:
    num_threads = min(os.cpu_count() or 4, 8)
    torch.set_num_threads(num_threads)
    print(f"Configured CPU Threads: {num_threads}")


## 2. Streaming Dataset Pipeline (Zero-OOM Out-of-Core Architecture)

To prevent Out-Of-Memory (OOM) failures when scaling up to millions of frames:
- **RadioML 2018 (`RadioMLStreamingDataset`)**: Rather than loading multi-gigabyte arrays into memory, we construct an index catalog table in RAM (<25 MB). Slices `X[offset]` of shape `(1024, 2)` are read directly on-the-fly per batch. Multiprocessing workers lazily open dedicated SWMR HDF5 handles to ensure thread safety and maximum I/O throughput.
- **Synthetic IQ (`SyntheticStreamingDataset`)**: Generates stochastic complex baseband IQ realizations with AWGN and CFO dynamically during training, achieving zero memory footprint and acting as natural data augmentation. Validation frames use deterministic seeding for consistent epoch-over-epoch evaluation.


In [ ]:
# Class definitions
RADIOML_24_CLASSES = [
    'OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK',
    '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', '32QAM', '64QAM',
    '128QAM', '256QAM', 'AM-SSB-WC', 'AM-SSB-SC', 'AM-DSB-WC', 'AM-DSB-SC',
    'FM', 'GFSK', 'CPFSK'
]

RADIOML_TO_PARENT_MAP = {
    'AM-SSB-WC': 'AM', 'AM-SSB-SC': 'AM', 'AM-DSB-WC': 'AM', 'AM-DSB-SC': 'AM',
    'FM': 'FM', 'BPSK': 'BPSK', 'GFSK': 'GFSK', 'CPFSK': 'GFSK',
    'QPSK': 'QPSK', '8PSK': '8PSK', '16PSK': '8PSK', '32PSK': '8PSK',
    '16QAM': '16QAM', '32QAM': '16QAM', '64QAM': '64QAM', '128QAM': '64QAM',
    '256QAM': '256QAM', 'OOK': 'ASK', '4ASK': 'ASK', '8ASK': 'ASK',
    '16APSK': '16APSK', '32APSK': '16APSK', '64APSK': '32APSK', '128APSK': '32APSK'
}

ALL_30_CLASSES = [
    # Core Classes
    "AM", "FM", "BPSK", "GFSK", "QPSK", "8PSK",
    "16QAM", "64QAM", "256QAM", "ASK", "16APSK", "32APSK",
    "OQPSK", "OFDM", "SC-FDMA", "Noise",
    # Granular Subclasses
    "OOK", "4ASK", "8ASK", "16PSK", "32PSK",
    "64APSK", "128APSK", "32QAM", "128QAM",
    "AM-SSB-WC", "AM-SSB-SC", "AM-DSB-WC", "AM-DSB-SC", "CPFSK"
]

def synthesize_raw_iq(mod_type, frame_len=1024, snr_db=15.0):
    """Synthesize complex baseband IQ samples for a given modulation class."""
    if mod_type == "Noise":
        sig = (np.random.randn(frame_len) + 1j * np.random.randn(frame_len)) / np.sqrt(2.0)
        return sig.astype(np.complex64)

    if mod_type in ["AM", "AM-DSB-WC", "AM-DSB-SC", "AM-SSB-WC", "AM-SSB-SC", "FM"]:
        t = np.arange(frame_len)
        f_m = np.random.uniform(0.002, 0.02)
        audio = np.sin(2 * np.pi * f_m * t)
        if mod_type in ["AM", "AM-DSB-WC"]:
            sig = 1.0 + 0.8 * audio + 1j * 0.0
        elif mod_type == "AM-DSB-SC":
            sig = audio + 1j * 0.0
        elif mod_type in ["AM-SSB-WC", "AM-SSB-SC"]:
            sig = (audio + 1j * np.cos(2 * np.pi * f_m * t)) + (1.0 if "WC" in mod_type else 0.0)
        else:
            sig = np.exp(1j * (2 * np.pi * 0.1 * np.cumsum(audio)))
    elif "PSK" in mod_type:
        m = 2 if mod_type == "BPSK" else (4 if mod_type == "QPSK" else (8 if mod_type == "8PSK" else 16))
        s = np.exp(1j * (2 * np.pi / m * np.random.randint(0, m, frame_len // 4)))
        sig = np.repeat(s, 4)[:frame_len]
    elif "QAM" in mod_type:
        m = 16 if mod_type == "16QAM" else (64 if mod_type == "64QAM" else 256)
        grid = np.linspace(-3, 3, int(np.sqrt(m)))
        c = (grid[:, None] + 1j * grid[None, :]).ravel()
        s = np.random.choice(c, frame_len // 4)
        sig = np.repeat(s, 4)[:frame_len]
    else:
        bits = np.random.choice([-1, 1], frame_len // 4)
        sig = np.repeat(bits + 1j * 0.0, 4)[:frame_len]

    cfo = np.random.uniform(-0.02, 0.02)
    sig = sig * np.exp(1j * 2 * np.pi * cfo * np.arange(frame_len))
    snr_linear = 10.0 ** (snr_db / 10.0)
    p_sig = np.mean(np.abs(sig)**2) + 1e-12
    p_noise = p_sig / snr_linear
    noise = (np.random.randn(frame_len) + 1j * np.random.randn(frame_len)) * np.sqrt(p_noise / 2.0)
    return (sig + noise).astype(np.complex64)


class RadioMLStreamingDataset(Dataset):
    """
    Streaming PyTorch Dataset for RadioML 2018.01A HDF5 files.
    Reads individual IQ frames on-the-fly without pre-loading arrays into memory.
    """
    def __init__(self, h5_path, indices, labels, snrs, frame_len=1024):
        self.h5_path = h5_path
        self.indices = np.ascontiguousarray(indices, dtype=np.int64)
        self.labels = np.ascontiguousarray(labels, dtype=np.int64)
        self.snrs = np.ascontiguousarray(snrs, dtype=np.float32)
        self.frame_len = frame_len
        self.h5_file = None
        self.X_ds = None

    def __len__(self):
        return len(self.indices)

    def _open_h5(self):
        if self.h5_file is None:
            import h5py
            self.h5_file = h5py.File(self.h5_path, 'r', swmr=True, libver='latest')
            self.X_ds = self.h5_file['X']

    def __getitem__(self, idx):
        if self.h5_file is None:
            self._open_h5()
        file_offset = int(self.indices[idx])
        raw = self.X_ds[file_offset]  # shape: (1024, 2)
        iq = np.transpose(raw, (1, 0)).astype(np.float32)  # shape: (2, 1024)
        
        # Power normalization
        p = np.mean(iq**2) * 2.0
        if p > 1e-12:
            iq = iq / np.sqrt(p)
            
        return torch.from_numpy(iq), self.labels[idx]


class SyntheticStreamingDataset(Dataset):
    """
    Streaming PyTorch Dataset for high-throughput synthetic IQ signal generation.
    Generates frames on-the-fly, consuming zero RAM for datasets of arbitrary size.
    """
    def __init__(self, class_indices, snrs, target_classes, frame_len=1024, is_train=True, seed_offset=0):
        self.class_indices = np.ascontiguousarray(class_indices, dtype=np.int64)
        self.snrs = np.ascontiguousarray(snrs, dtype=np.float32)
        self.target_classes = target_classes
        self.frame_len = frame_len
        self.is_train = is_train
        self.seed_offset = seed_offset

    def __len__(self):
        return len(self.class_indices)

    def __getitem__(self, idx):
        c_idx = int(self.class_indices[idx])
        c_name = self.target_classes[c_idx]
        snr = float(self.snrs[idx])

        # Deterministic seed for reproducible validation set
        if not self.is_train:
            rng_state = np.random.get_state()
            np.random.seed((self.seed_offset + idx * 7919) % (2**31 - 1))
            frame = synthesize_raw_iq(c_name, frame_len=self.frame_len, snr_db=snr)
            np.random.set_state(rng_state)
        else:
            frame = synthesize_raw_iq(c_name, frame_len=self.frame_len, snr_db=snr)

        # Power normalization
        p = np.mean(np.abs(frame)**2)
        if p > 1e-12:
            frame = frame / np.sqrt(p)

        iq = np.empty((2, self.frame_len), dtype=np.float32)
        iq[0, :] = frame.real.astype(np.float32)
        iq[1, :] = frame.imag.astype(np.float32)
        return torch.from_numpy(iq), c_idx


def worker_init_fn(worker_id):
    """Ensure worker processes lazily open clean HDF5 handles."""
    worker_info = torch.utils.data.get_worker_info()
    if worker_info is not None:
        dataset = worker_info.dataset
        if hasattr(dataset, '_open_h5'):
            dataset._open_h5()


def stratified_split(labels, test_size=0.20, random_state=42):
    """Stratified train/validation index partition with zero-dependency fallback."""
    try:
        from sklearn.model_selection import train_test_split
        return train_test_split(
            np.arange(len(labels)),
            test_size=test_size,
            random_state=random_state,
            stratify=labels
        )
    except ImportError:
        rng = np.random.RandomState(random_state)
        labels_arr = np.array(labels)
        train_indices, val_indices = [], []
        for c in np.unique(labels_arr):
            c_indices = np.where(labels_arr == c)[0]
            rng.shuffle(c_indices)
            n_val = int(round(len(c_indices) * test_size))
            val_indices.extend(c_indices[:n_val])
            train_indices.extend(c_indices[n_val:])
        rng.shuffle(train_indices)
        rng.shuffle(val_indices)
        return np.array(train_indices), np.array(val_indices)


SNR_LEVELS = list(range(-20, 32, 2))  # 26 SNR steps: -20, -18, ..., +30 dB
FRAMES_PER_BLOCK = 4096

if DATASET_CHOICE_UPPER == "RADIOML":
    TARGET_CLASSES = RADIOML_24_CLASSES
    # Determine frames per (class, SNR)
    if SAMPLES_PER_CLASS is not None and SAMPLES_PER_CLASS > 0:
        frames_per_snr = min(FRAMES_PER_BLOCK, max(1, SAMPLES_PER_CLASS // len(SNR_LEVELS)))
    else:
        frames_per_snr = FRAMES_PER_BLOCK

    print(f"Building RadioML index catalog: {len(TARGET_CLASSES)} classes x {len(SNR_LEVELS)} SNRs x {frames_per_snr} frames/bin...")
    sample_indices = []
    sample_labels = []
    sample_snrs = []

    for c_idx, c_name in enumerate(TARGET_CLASSES):
        for s_idx, snr_val in enumerate(SNR_LEVELS):
            block_offset = (c_idx * 26 + s_idx) * FRAMES_PER_BLOCK
            offsets = np.arange(block_offset, block_offset + frames_per_snr, dtype=np.int64)
            sample_indices.extend(offsets)
            sample_labels.extend([c_idx] * frames_per_snr)
            sample_snrs.extend([snr_val] * frames_per_snr)

    total_frames = len(sample_indices)
    print(f"RadioML catalog built: {total_frames:,} total frames indexed (<25 MB metadata index in RAM).")

    # Stratified Train/Val Split on index tables
    train_idx, val_idx = stratified_split(
        sample_labels,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED
    )

    train_indices = [sample_indices[i] for i in train_idx]
    train_labels = [sample_labels[i] for i in train_idx]
    train_snrs = [sample_snrs[i] for i in train_idx]

    val_indices = [sample_indices[i] for i in val_idx]
    val_labels = [sample_labels[i] for i in val_idx]
    val_snrs = [sample_snrs[i] for i in val_idx]

    train_dataset = RadioMLStreamingDataset(dataset_path, train_indices, train_labels, train_snrs, frame_len=FRAME_LEN)
    val_dataset = RadioMLStreamingDataset(dataset_path, val_indices, val_labels, val_snrs, frame_len=FRAME_LEN)

else:  # GENERATED
    TARGET_CLASSES = ALL_30_CLASSES
    samples_per_class = SAMPLES_PER_CLASS if (SAMPLES_PER_CLASS is not None and SAMPLES_PER_CLASS > 0) else 2000
    total_frames = len(TARGET_CLASSES) * samples_per_class
    print(f"Building Synthetic index catalog: {len(TARGET_CLASSES)} classes x {samples_per_class} frames/class = {total_frames:,} frames...")

    gen_classes = []
    gen_snrs = []
    discrete_snrs = np.arange(-10.0, 32.0, 2.0)  # Discrete SNR range for clean SNR curves

    for c_idx in range(len(TARGET_CLASSES)):
        for i in range(samples_per_class):
            gen_classes.append(c_idx)
            gen_snrs.append(float(discrete_snrs[i % len(discrete_snrs)]))

    # Stratified Train/Val Split
    train_idx, val_idx = stratified_split(
        gen_classes,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED
    )

    train_c = [gen_classes[i] for i in train_idx]
    train_s = [gen_snrs[i] for i in train_idx]
    val_c = [gen_classes[i] for i in val_idx]
    val_s = [gen_snrs[i] for i in val_idx]

    train_dataset = SyntheticStreamingDataset(train_c, train_s, TARGET_CLASSES, frame_len=FRAME_LEN, is_train=True)
    val_dataset = SyntheticStreamingDataset(val_c, val_s, TARGET_CLASSES, frame_len=FRAME_LEN, is_train=False, seed_offset=12345)

snr_val = np.array(val_dataset.snrs, dtype=np.float32)

print(f"Dataset split: {len(train_dataset):,} Training frames (80%), {len(val_dataset):,} Validation frames (20%)")

# Initialize Streaming DataLoaders
effective_workers = NUM_WORKERS if device.type == "cuda" else 0
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=effective_workers,
    worker_init_fn=worker_init_fn,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=effective_workers,
    worker_init_fn=worker_init_fn,
    pin_memory=(device.type == "cuda"),
    drop_last=False
)

print(f"Streaming DataLoaders ready: Batch Size = {BATCH_SIZE}, Train Steps = {len(train_loader):,}, Val Steps = {len(val_loader):,}")


## 3. 1D ResNet Architecture (ResNet1D) Directly Operating on Raw IQ

The **1D ResNet (ResNet1D)** processes raw 2-channel In-phase and Quadrature signals directly:
- **Stem**: Conv1d(2, 32, k=7, p=3) with Batch Normalization and ReLU.
- **Residual Blocks**: 4 stages (32 -> 32 -> 64 -> 128 -> 128 channels) with identity shortcut connections and projection shortcuts for downsampling.
- **Global Pooling & Head**: AdaptiveAvgPool1d(1) -> Linear(128, num_classes).
- **Dynamic Dimensions**: Accommodates variable sequence lengths and batch sizes without structural modification.


In [ ]:
class Residual1DBlock(nn.Module):
    """1D Residual Convolutional Block for complex IQ signals."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv1d(
            in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(
            out_channels, out_channels, kernel_size=7, stride=1, padding=3, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        return self.relu(out)


class ResNet1D(nn.Module):
    """
    1D ResNet Architecture for Automatic Modulation Classification.
    Ingests raw 2-channel [Batch, 2, SequenceLength] IQ tensors directly.
    """
    def __init__(self, num_classes: int, in_channels: int = 2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True)
        )
        self.stage1 = Residual1DBlock(32, 32, stride=1)
        self.stage2 = Residual1DBlock(32, 64, stride=2)
        self.stage3 = Residual1DBlock(64, 128, stride=2)
        self.stage4 = Residual1DBlock(128, 128, stride=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.stem(x)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        out = self.pool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)


# Instantiate Model
num_classes = len(TARGET_CLASSES)
model = ResNet1D(num_classes=num_classes, in_channels=2).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Instantiated 1D ResNet ({num_classes} classes): {total_params:,} trainable parameters.")
print(model)


## 4. GPU-Accelerated Training Loop with Mixed Precision (AMP)

We train the 1D ResNet using:
- **Optimizer**: AdamW (`weight_decay=1e-4`)
- **Learning Rate Schedule**: Cosine Annealing scheduler (`T_max=EPOCHS`)
- **Automatic Mixed Precision (AMP)**: Enabled on CUDA to accelerate convolution math and reduce GPU memory consumption.
- **Best Model Checkpoint Tracking**: Evaluates validation accuracy after every epoch and saves the best model state dictionary.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_acc = 0.0
best_model_state = None

print(f"Starting 1D ResNet training on {device} ({'AMP enabled' if use_amp else 'FP32'}) for {EPOCHS} epochs...")
import time
start_time = time.time()

pbar = tqdm(range(1, EPOCHS + 1), desc="Training 1D ResNet")
for epoch in pbar:
    # Training Phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for bx, by in train_loader:
        bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(bx)
            loss = criterion(logits, by)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * bx.size(0)
        preds = torch.argmax(logits, dim=1)
        correct_train += (preds == by).sum().item()
        total_train += bx.size(0)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    train_epoch_loss = running_loss / max(total_train, 1)
    train_epoch_acc = (correct_train / max(total_train, 1)) * 100.0

    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(bx)
                loss = criterion(logits, by)

            running_val_loss += loss.item() * bx.size(0)
            preds = torch.argmax(logits, dim=1)
            correct_val += (preds == by).sum().item()
            total_val += bx.size(0)

    val_epoch_loss = running_val_loss / max(total_val, 1)
    val_epoch_acc = (correct_val / max(total_val, 1)) * 100.0

    history['epoch'].append(epoch)
    history['train_loss'].append(train_epoch_loss)
    history['train_acc'].append(train_epoch_acc)
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)
    history['lr'].append(current_lr)

    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        best_model_state = model.state_dict().copy()

    pbar.set_postfix({
        'loss': f"{train_epoch_loss:.3f}",
        'acc': f"{train_epoch_acc:.1f}%",
        'val_loss': f"{val_epoch_loss:.3f}",
        'val_acc': f"{val_epoch_acc:.1f}%"
    })

    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        pbar.write(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | "
                   f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_epoch_acc:5.2f}% | "
                   f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:5.2f}% | "
                   f"LR: {current_lr:.6f}")

total_duration = time.time() - start_time
print("=" * 70)
print(f"Training completed in {total_duration:.2f}s ({total_duration/EPOCHS:.2f}s/epoch).")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Loaded best model state checkpoint.")

progress_df = pd.DataFrame(history)
print("\n" + "=" * 30 + " TOP 5 BEST EPOCHS " + "=" * 30)
print(progress_df.sort_values(by="val_acc", ascending=False).head(5).to_string(index=False))


## 5. Visualization: Training & Validation Learning Curves wrt Epoch

Plot the learning dynamics of training/validation loss and classification accuracy over epochs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs_arr = history['epoch']
best_epoch_idx = int(np.argmax(history['val_acc']))
best_epoch = epochs_arr[best_epoch_idx]

# Subplot 1: Loss vs Epoch
axes[0].plot(epochs_arr, history['train_loss'], 'o-', color='#1f77b4', linewidth=2, label='Training Loss')
axes[0].plot(epochs_arr, history['val_loss'], 's-', color='#ff7f0e', linewidth=2, label='Validation Loss')
axes[0].axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best Val Epoch ({best_epoch})')
axes[0].set_title('1D ResNet Loss vs Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=11)
axes[0].legend(loc='upper right', frameon=True)
axes[0].grid(True, alpha=0.3)

# Subplot 2: Accuracy vs Epoch
axes[1].plot(epochs_arr, history['train_acc'], 'o-', color='#2ca02c', linewidth=2, label='Training Accuracy')
axes[1].plot(epochs_arr, history['val_acc'], 's-', color='#d62728', linewidth=2, label='Validation Accuracy')
axes[1].axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best Val Acc ({best_val_acc:.2f}%)')
axes[1].scatter([best_epoch], [best_val_acc], color='#d62728', s=120, zorder=5)
axes[1].set_title('1D ResNet Accuracy vs Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].legend(loc='lower right', frameon=True)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("amc_resnet1d_training_curves.png", dpi=300)
plt.show()
print("Saved plot to: amc_resnet1d_training_curves.png")


## 6. Validation Evaluation & Confusion Matrix Heatmap

Evaluate model classification accuracy across all target modulation classes and generate a normalized confusion matrix heatmap.


In [ ]:
model.eval()
val_preds_list = []
val_targets_list = []

with torch.no_grad():
    for bx, by in val_loader:
        bx = bx.to(device)
        logits = model(bx)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        val_preds_list.extend(preds)
        val_targets_list.extend(by.numpy())

val_preds = np.array(val_preds_list)
val_targets = np.array(val_targets_list)

exact_val_acc = (val_preds == val_targets).mean() * 100.0
print(f"Overall Exact Top-1 Accuracy: {exact_val_acc:.2f}%")
print("-" * 65)

# Classification Report
try:
    from sklearn.metrics import classification_report, confusion_matrix
    print(classification_report(val_targets, val_preds, target_names=TARGET_CLASSES, digits=3))
    cm = confusion_matrix(val_targets, val_preds, labels=range(len(TARGET_CLASSES)), normalize='true')
except ImportError:
    # Pure numpy confusion matrix fallback
    num_c = len(TARGET_CLASSES)
    cm = np.zeros((num_c, num_c), dtype=np.float32)
    for t, p in zip(val_targets, val_preds):
        cm[t, p] += 1
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    cm = cm / row_sums

# Confusion Matrix Heatmap
plt.figure(figsize=(14, 12))
if sns_available:
    import seaborn as sns
    sns.heatmap(
        cm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=TARGET_CLASSES,
        yticklabels=TARGET_CLASSES,
        cbar=True
    )
else:
    plt.imshow(cm, cmap='Blues', aspect='auto')
    plt.colorbar()
    plt.xticks(range(len(TARGET_CLASSES)), TARGET_CLASSES, rotation=45, ha='right')
    plt.yticks(range(len(TARGET_CLASSES)), TARGET_CLASSES)

plt.title(f'1D ResNet AMC Confusion Matrix (Overall Val Acc: {exact_val_acc:.1f}%)', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Modulation', fontsize=11)
plt.ylabel('True Modulation', fontsize=11)
plt.tight_layout()
plt.savefig("amc_resnet1d_confusion_matrix.png", dpi=300)
plt.show()


## 7. Classification Accuracy vs Signal-to-Noise Ratio (SNR)

Measure how classification accuracy scales with received SNR (-20 dB to +30 dB).


In [ ]:
snrs_unique = np.sort(np.unique(snr_val))
exact_acc_by_snr = []

for s in snrs_unique:
    mask = (snr_val == s)
    sub_preds = val_preds[mask]
    sub_targets = val_targets[mask]
    if len(sub_targets) == 0:
        continue
    exact = (sub_preds == sub_targets).mean() * 100.0
    exact_acc_by_snr.append(exact)

plt.figure(figsize=(10, 6))
plt.plot(snrs_unique, exact_acc_by_snr, 'o-', color='#1f77b4', linewidth=2.5, label='1D ResNet Exact Top-1 Accuracy')
plt.axhline(90, color='red', linestyle='--', alpha=0.7, label='90% Target')
plt.title('1D ResNet DL-AMC Accuracy vs SNR', fontsize=13, fontweight='bold')
plt.xlabel('Signal-to-Noise Ratio (SNR in dB)', fontsize=11)
plt.ylabel('Classification Accuracy (%)', fontsize=11)
plt.ylim(0, 105)
plt.legend(loc='lower right', frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("amc_resnet1d_accuracy_vs_snr.png", dpi=300)
plt.show()


## 8. Export Trained 1D ResNet to Portable ONNX & Remove PyTorch Weights

Export the trained model to high-performance portable ONNX format with dynamic axes (`batch_size`, `sequence_length`). Once exported and validated, remove intermediate PyTorch `.pt` checkpoint.


In [ ]:
import onnx
import onnxruntime as ort

pt_output_path = "amc_resnet1d.pt"
onnx_output_path = "amc_resnet1d.onnx"

model.eval()

# 1. Save PyTorch state dictionary checkpoint
torch.save(model.state_dict(), pt_output_path)
print(f"Saved PyTorch model checkpoint: {pt_output_path}")

# 2. Export portable ONNX with dynamic batch and sequence length
cpu_model = model.to("cpu")
dummy_input = torch.randn(1, 2, FRAME_LEN, dtype=torch.float32)

torch.onnx.export(
    cpu_model,
    dummy_input,
    onnx_output_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size", 2: "sequence_length"},
        "output": {0: "batch_size"}
    }
)
print(f"Successfully exported ONNX artifact to: {onnx_output_path}")

# 3. Verify ONNX model integrity
onnx_proto = onnx.load(onnx_output_path)
onnx.checker.check_model(onnx_proto)
print("ONNX model structure validated successfully.")

# 4. Benchmark ONNX Runtime latency
session = ort.InferenceSession(onnx_output_path, providers=["CPUExecutionProvider"])
test_iq = np.random.randn(1, 2, FRAME_LEN).astype(np.float32)

# Warmup
for _ in range(20):
    _ = session.run(None, {"input": test_iq})

t0 = time.perf_counter()
N_RUNS = 200
for _ in range(N_RUNS):
    out = session.run(None, {"input": test_iq})
t1 = time.perf_counter()

avg_latency_ms = ((t1 - t0) / N_RUNS) * 1000.0
print(f"Average ONNX Inference Latency: {avg_latency_ms:.3f} ms / frame ({1000.0/avg_latency_ms:.1f} frames/sec)")
print(f"Output prediction logits shape: {out[0].shape}")

# 5. Remove intermediate PyTorch checkpoint (retain only production ONNX)
if os.path.exists(pt_output_path):
    os.remove(pt_output_path)
    print(f"Cleaned up checkpoint: removed '{pt_output_path}' (retaining only production '{onnx_output_path}')")


## 9. Summary & Deployment Instructions

### Generated Artifacts (in `/kaggle/working/`):
- `amc_resnet1d.onnx` (~2.0 MB): Production ONNX artifact ready to drop directly into `gr_playground/dsp/models/amc_resnet1d.onnx`.
- `amc_resnet1d_training_curves.png`: Loss and accuracy curves across training epochs.
- `amc_resnet1d_confusion_matrix.png`: Normalized confusion matrix heatmap.
- `amc_resnet1d_accuracy_vs_snr.png`: Classification accuracy curve as a function of SNR (-20 to +30 dB).

### Deployment to gr-playground:
```bash
cp amc_resnet1d.onnx gr_playground/dsp/models/amc_resnet1d.onnx
```
